# Claude Token & Cost Estimator

**Note:** This tool was originally created by Anthropic but I am trying to expand, improve, and keep it up to date with the latest models and their cost. If you want to contribute, let me know so we can collaborate.

Everything after this line, in this markdown file, was here originally.

-----------------------------

This notebook uses Anthropic's official **`/v1/messages/count_tokens`** endpoint to get an *exact* token count for a prompt against a specific Claude model, before you actually send it for generation. Counting tokens this way is free and doesn't count against your normal message rate limits.

It then estimates $ cost using current published per-token pricing (input cost is exact since it's based on the real count; output cost is an *estimate* since you choose how many output tokens to assume).

**Note on pricing:** rates below are current as of July 2026, but Anthropic can change them. If your numbers look off, check https://www.anthropic.com/pricing before trusting the cost estimate.

**Note on Sonnet 5 pricing:** it's on introductory pricing ($2/$10 per million tokens) through Aug 31, 2026, then standard pricing ($3/$15) from Sept 1, 2026. There's a toggle below for this.

## 1. Setup

In [1]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:00


In [2]:
import getpass
import anthropic

api_key = getpass.getpass("Paste your Anthropic API key (input hidden): ")
client = anthropic.Anthropic(api_key=api_key)
print("Client ready.")

Paste your Anthropic API key (input hidden): ··········
Client ready.


## 2. Pricing table (per million tokens, USD)

Edit this dict if Anthropic updates pricing — treat it as the single source of truth for the rest of the notebook.

In [3]:
# model_id: (input $ / MTok, output $ / MTok)
PRICING = {
    "claude-fable-5":(10.00, 50.00),
    "claude-opus-5":(5.00, 25.00),
    "claude-opus-4-8":(5.00, 25.00),
    "claude-sonnet-5":(3.00, 15.00),   # standard pricing, from Sept 1 2026
    "claude-sonnet-5-intro":(2.00, 10.00),   # introductory pricing through Aug 31 2026 (same model id, different rate)
    "claude-haiku-4-5-20251001":(1.00, 5.00),
}

MODEL_CHOICES = {
    "1": ("Claude Fable 5", "claude-fable-5"),
    "2": ("Claude Opus 5", "claude-opus-5"),
    "3": ("Claude Opus 4.8", "claude-opus-4-8"),
    "4": ("Claude Sonnet 5 (standard pricing, $3/$15)", "claude-sonnet-5"),
    "5": ("Claude Sonnet 5 (introductory pricing thru Aug 31 2026, $2/$10)", "claude-sonnet-5"),
    "6": ("Claude Haiku 4.5", "claude-haiku-4-5-20251001"),
}

print("Pick a model:")
for k, (label, _) in MODEL_CHOICES.items():
    print(f"  {k}. {label}")

Pick a model:
  1. Claude Fable 5
  2. Claude Opus 5
  3. Claude Opus 4.8
  4. Claude Sonnet 5 (standard pricing, $3/$15)
  5. Claude Sonnet 5 (introductory pricing thru Aug 31 2026, $2/$10)
  6. Claude Haiku 4.5


## 3. Choose model + enter your prompt

In [ ]:
choice = input("Enter the number for your model choice (1-6): ").strip()
label, model_id = MODEL_CHOICES.get(choice, MODEL_CHOICES["2"])
pricing_key = model_id if choice != "3" else "claude-sonnet-5-intro"
print(f"Using: {label}  (model id sent to API: {model_id})")

In [ ]:
# Option A: type/paste your prompt directly here
prompt_text = """

"""

# Option B: uncomment to load a prompt from a local .txt file instead
# with open("my_prompt.txt", "r") as f:
#     prompt_text = f.read()

print(f"Prompt loaded ({len(prompt_text)} characters).")

Prompt loaded (18889 characters).


## 4. Optional: system prompt
System prompt tokens count too — include one here if your real request will use one. Leave blank if not.

In [ ]:
system_prompt = ""  # leave as empty string if you don't use one

## 5. Count tokens (exact, via the API — free, no generation happens)

In [ ]:
kwargs = {
    "model": model_id,
    "messages": [{"role": "user", "content": prompt_text}],
}
if system_prompt.strip():
    kwargs["system"] = system_prompt

response = client.messages.count_tokens(**kwargs)
input_tokens = response.input_tokens
print(f"Input tokens for this prompt on {label}: {input_tokens}")

Input tokens for this prompt on Claude Opus 5: 4969


## 6. Estimate cost

Input cost is exact (based on the real count above). Output cost is a guess — enter roughly how many tokens you expect Claude to generate in response (a rough rule of thumb is ~0.75 tokens per word for plain English prose; more for code).

In [ ]:
estimated_output_tokens = 500  # <-- change this to whatever you expect the response length to be

input_rate, output_rate = PRICING[pricing_key]
input_cost = (input_tokens / 1_000_000) * input_rate
output_cost = (estimated_output_tokens / 1_000_000) * output_rate
total_cost = input_cost + output_cost

print(f"Model: {label}")
print(f"Input tokens:  {input_tokens:>8}  ->  ${input_cost:.6f}")
print(f"Output tokens: {estimated_output_tokens:>8} (assumed) -> ${output_cost:.6f}")
print(f"Estimated total cost: ${total_cost:.6f}")

Model: Claude Opus 5
Input tokens:      4969  ->  $0.024845
Output tokens:      500 (assumed) -> $0.012500
Estimated total cost: $0.037345


## 7. (Optional) Batch-check multiple prompts at once

Useful if you want to estimate cost across several prompts before running a batch (e.g. many podcast script drafts, citation checks, etc.).

In [ ]:
prompts = [
    "First prompt text goes here.",
    "Second prompt text goes here.",
    # add as many as you want
]

assumed_output_tokens_each = 500

total_input_tokens = 0
print(f"{'#':<4}{'input tokens':<15}{'est. cost':<12}")
for i, p in enumerate(prompts, 1):
    r = client.messages.count_tokens(model=model_id, messages=[{"role": "user", "content": p}])
    total_input_tokens += r.input_tokens
    cost = (r.input_tokens / 1_000_000) * input_rate + (assumed_output_tokens_each / 1_000_000) * output_rate
    print(f"{i:<4}{r.input_tokens:<15}${cost:.6f}")

grand_total_cost = (total_input_tokens / 1_000_000) * input_rate + (len(prompts) * assumed_output_tokens_each / 1_000_000) * output_rate
print(f"\nTotal input tokens across {len(prompts)} prompts: {total_input_tokens}")
print(f"Total estimated cost: ${grand_total_cost:.6f}")